# Laboratorio 01 — Pipeline Declarativa Básica con Lakeflow

**Semana:** 05 | **Actividad de referencia:** Actividad 01  
**Modalidad:** Individual | **Entorno:** Databricks Lakeflow (Spark Declarative Pipelines)

---

## Instrucciones generales

Crea una pipeline declarativa básica con `pyspark.pipelines` (`dp`) para tu dataset propio. La pipeline debe incluir al menos una `materialized_view` (batch) y una `table` (streaming o batch), además de demostrar el uso de parámetros de pipeline.

> **Importante:** Este notebook debe ejecutarse desde un Databricks Pipeline (Lakeflow), no directamente como notebook interactivo. Las celdas de código son la definición de la pipeline.

## Parte 1 — Descripción del dataset y diseño de la pipeline

1. **Nombre, fuente y URL** del dataset.
2. **¿Qué representará la materialized_view?** ¿Un resumen, un join, una vista limpia de los datos?
3. **¿Qué representará la table?** ¿La carga incremental, los datos filtrados, un agrupamiento?
4. **Parámetros de pipeline:** ¿Qué valores querrías parametrizar? (ruta de origen, nombres de tabla, umbrales, etc.)
5. **Preguntas de negocio** que esta pipeline ayudaría a responder.

**Escribe tu respuesta aquí:**

## Parte 2 — Importaciones y parámetros de pipeline

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# Parámetros de pipeline: se configuran en la UI de Lakeflow (Settings → Parameters)
# Aquí solo se leen; los valores se pasan en el job de la pipeline
RUTA_ORIGEN  = spark.conf.get("ruta_origen",  "/Volumes/workspace/default/week_5/tu_archivo.csv")
ENTORNO      = spark.conf.get("entorno",       "dev")
UMBRAL_MIN   = float(spark.conf.get("umbral_minimo", "0"))

print(f"Parámetros de pipeline:")
print(f"  ruta_origen   = {RUTA_ORIGEN}")
print(f"  entorno       = {ENTORNO}")
print(f"  umbral_minimo = {UMBRAL_MIN}")

## Parte 3 — Perfil técnico del dataset

> Ejecuta esta sección como notebook interactivo (fuera de la pipeline) para entender el dataset antes de definir las transformaciones declarativas.

In [ ]:
# Exploración interactiva — NO usar decoradores @dp aquí
df_explorar = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(RUTA_ORIGEN)

print(f"Filas: {df_explorar.count():,} | Columnas: {len(df_explorar.columns)}")
df_explorar.printSchema()
df_explorar.show(5, truncate=False)

In [ ]:
# Nulos por columna
total = df_explorar.count()
df_explorar.select([
    F.round(
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) * 100.0 / total, 1
    ).alias(f"{c}_pct_nulos")
    for c in df_explorar.columns
]).show(truncate=False)

In [ ]:
# Cardinalidades de columnas categóricas
# Reemplaza 'columna_categorica' con el nombre real de tu columna
df_explorar.groupBy("columna_categorica").count().orderBy(F.col("count").desc()).show(15)

**Observaciones del perfil:**  
¿Qué columnas tienen nulos que deberías manejar en la pipeline? ¿Qué columna categórica tiene mejor distribución para usarla como partición?

## Parte 4 — Definición declarativa de la pipeline

Estas celdas constituyen la pipeline en sí. Cuando se ejecuten desde Lakeflow, `@dp.materialized_view()` y `@dp.table()` registran las transformaciones en el grafo de la pipeline.

In [ ]:
# ─────────────────────────────────────────────────────────
# Bronze: lectura raw — materialized_view
# ─────────────────────────────────────────────────────────
@dp.materialized_view(
    comment="Carga raw del dataset desde el volumen. Sin transformaciones."
)
def bronze_mi_dataset():
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(RUTA_ORIGEN)
        .withColumn("_ingest_ts", F.current_timestamp())
        .withColumn("_entorno",   F.lit(ENTORNO))
    )

In [ ]:
# ─────────────────────────────────────────────────────────
# Silver: limpieza y filtrado — materialized_view
# ─────────────────────────────────────────────────────────
@dp.materialized_view(
    comment="Silver: registros filtrados, tipos correctos, strings normalizados."
)
def silver_mi_dataset():
    return (
        dp.read("bronze_mi_dataset")
        .filter(F.col("columna_clave").isNotNull())
        .filter(F.col("columna_numerica") >= UMBRAL_MIN)
        .withColumn("columna_texto", F.trim(F.lower(F.col("columna_texto"))))
        # Añade aquí tus propias transformaciones de limpieza
        .withColumn("_silver_ts", F.current_timestamp())
    )

In [ ]:
# ─────────────────────────────────────────────────────────
# Gold: agregación de negocio — table (persistida)
# ─────────────────────────────────────────────────────────
@dp.table(
    comment="Gold: KPIs por categoría calculados desde Silver."
)
def gold_kpis_mi_dataset():
    return (
        dp.read("silver_mi_dataset")
        .groupBy("columna_categorica")
        .agg(
            F.count("*").alias("total_registros"),
            F.avg("columna_numerica").alias("promedio"),
            F.max("columna_numerica").alias("maximo"),
            F.min("columna_numerica").alias("minimo")
        )
        .orderBy(F.col("total_registros").desc())
    )

**Análisis de la pipeline:**
1. ¿Cuántos nodos tiene el grafo de tu pipeline (Bronze → Silver → Gold)?
2. ¿Cuál es la diferencia entre `@dp.materialized_view()` y `@dp.table()`? ¿Cuándo usarías cada uno?
3. ¿Qué pasa si falla `silver_mi_dataset`? ¿La pipeline intenta ejecutar `gold_kpis_mi_dataset` de todas formas?

## Parte 5 — Configuración de la pipeline (teórico)

Describe en markdown cómo configurarías esta pipeline en la UI de Lakeflow.

```json
{
  "name": "lab05-01-pipeline-mi-dataset",
  "target": "workspace.default",
  "clusters": [{"num_workers": 1}],
  "libraries": [
    {"notebook": {"path": "/semana_05/laboratorios/lab_01_pipeline_basica"}}
  ],
  "configuration": {
    "ruta_origen":    "/Volumes/workspace/default/week_5/tu_archivo.csv",
    "entorno":        "dev",
    "umbral_minimo":  "0"
  },
  "mode": "TRIGGERED"
}
```

**Preguntas:**
1. ¿Qué diferencia hay entre modo `TRIGGERED` y `CONTINUOUS`?
2. ¿Cómo cambia la config si quisieras procesar datos en tiempo real con Auto Loader?

## Parte 6 — Preguntas de negocio sobre los resultados Gold

Después de ejecutar la pipeline, consulta los resultados desde un notebook interactivo separado.

In [ ]:
# Ejecutar esta celda en un notebook interactivo DESPUÉS de correr la pipeline
# spark.table("workspace.default.gold_kpis_mi_dataset").show(15, truncate=False)

**Preguntas de negocio** (responde después de ejecutar la pipeline):
1. ¿Qué categoría tiene el mayor promedio? ¿Lo esperabas?
2. ¿Cuántos registros se perdieron de Bronze a Silver? ¿Por qué?
3. ¿Qué hallazgo en Gold te daría más impacto si lo presentaras a un stakeholder?

## Parte 7 — Reflexión final

1. ¿Qué ventaja tiene definir la pipeline con decoradores vs con un script imperativo que llama a `.write()`?
2. ¿Cómo maneja Lakeflow las dependencias entre `materialized_view` y `table` automáticamente?
3. ¿Qué pasa si modificas la lógica de `silver_mi_dataset` y re-ejecutas la pipeline? ¿Recomputa Gold también?
4. ¿Cómo agregarías una expectativa (`@dp.expect`) para detectar registros de baja calidad sin detener la pipeline?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_05/laboratorios/lab_01_pipeline_basica.ipynb semana_05/laboratorios/<tu-nombre>/lab_01_pipeline_basica.ipynb

git add semana_05/laboratorios/<tu-nombre>/lab_01_pipeline_basica.ipynb
git commit -m "lab: semana05 lab01 pipeline basica materialized_view table <nombre-dataset> - <tu-nombre>"
git push origin develop
```